# Manual Fabric workspace publication

Use with docs/manual-portal-deployment.md. Prepare fresh inputs with scripts/prepare_manual_fabric_assets.py and upload them through the Lakehouse UI. Attach the intended Lakehouse, set the published Environment ID, expected workspace name, and preparation SHA-256. SJD IDs are discovered only after the six jobs exist, then retained in Files/deployment/deployment.json. Run cells separately, in order. This notebook does not enable schedules or notifications and does not approve SDK release admission.


## 1. Authenticate and verify prepared inputs

The example IDs refer only to the walkthrough workspace. The current-workspace check fails closed elsewhere. The notebook uses the checked-in REST client and publication helpers, not an installed inference SDK.


In [ ]:
import notebookutils
import base64, gzip, hashlib, json, time
from pathlib import Path
TARGET_WORKSPACE_ID = notebookutils.runtime.context["defaultLakehouseWorkspaceId"]
TARGET_WORKSPACE_NAME = "<expected-workspace-name>"
TARGET_LAKEHOUSE_ID = notebookutils.runtime.context["defaultLakehouseId"]
TARGET_ENVIRONMENT_ID = "<published-environment-uuid>"
DEPLOYMENT_STAGE = "dev"
SJD_NAME_PREFIX = "pc_sjd_dev_"
DEPLOYMENT_FILES_ROOT = "Files/people-counter/sjd/v1/"
EXPECTED_INPUT_SHA256 = "REPLACE_WITH_PREPARATION_SHA256"
INPUT_FILE_NAME = "manual-publication.json"
assert notebookutils.runtime.context["currentWorkspaceId"] == TARGET_WORKSPACE_ID
root_uri = f"abfss://{TARGET_WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/{TARGET_LAKEHOUSE_ID}/Files"
notebookutils.fs.cp(root_uri + "/" + INPUT_FILE_NAME, "file:/tmp/manual-publication.json.gz", True)
raw = Path("/tmp/manual-publication.json.gz").read_bytes()
if INPUT_FILE_NAME.endswith(".gz"):
    raw = gzip.decompress(raw)
assert hashlib.sha256(raw).hexdigest() == EXPECTED_INPUT_SHA256, "Prepared input hash differs"
inputs = json.loads(raw)
assert inputs["schema"] == "people-counter-manual-publication-inputs-v1"
if "deploymentTemplates" not in inputs:
    raise ValueError("Regenerate the checked administrative inputs with the portable SDK sources")
exec(compile(inputs["clientSource"], "checked_fabric_rest.py", "exec"), globals())
exec(compile(inputs["publicationSource"], "checked_publication.py", "exec"), globals())
class NotebookTokenProvider:
    def get_token(self):
        return notebookutils.credentials.getToken("pbi")
client = FabricRESTController(NotebookTokenProvider(), workspace_id=TARGET_WORKSPACE_ID)
print("Prepared input SHA256:", hashlib.sha256(raw).hexdigest())
workspace_readback = client._json("GET", f"/workspaces/{TARGET_WORKSPACE_ID}")
if workspace_readback["displayName"] != TARGET_WORKSPACE_NAME:
    raise ValueError("Workspace name does not match the target ID")
print("Workspace verified:", workspace_readback["displayName"])


## 2. Provision managed monitoring

After this cell, use Workspace settings > Monitoring > Manage settings to enable diagnostic collection. Keep AI investigations and custom endpoint export disabled unless separately approved. Retain legacy artifacts and history. A successful definition update does not establish that collection is enabled.


In [ ]:
monitoring_name = "Monitoring_item_" + TARGET_WORKSPACE_NAME
monitoring_id = ensure_item(client, monitoring_name, "Monitoring")
print("Managed monitoring item:", monitoring_id)
monitoring_definition = client._lro_json("POST", f"/workspaces/{TARGET_WORKSPACE_ID}/items/{monitoring_id}/getDefinition")["definition"]
properties = {"$schema":"https://developer.microsoft.com/json-schemas/fabric/item/monitoring/definition/monitoringProperties/1.0.0/schema.json","dataCollectionSettings":[{"sources":{"workspaceDiagnostics":{"mode":"All"}},"destinations":{"eventhouse":{"mode":"Auto"},"eventStream":{"customEndpoint":{"enabled":False}}}}]}
parts = [p for p in monitoring_definition["parts"] if p["path"] == "MonitoringProperties.json"]
assert len(parts) == 1, "Monitoring properties part is ambiguous"
parts[0]["payload"] = base64.b64encode(json.dumps(properties).encode()).decode()
client._lro_json("POST", f"/workspaces/{TARGET_WORKSPACE_ID}/items/{monitoring_id}/updateDefinition", value={"definition":monitoring_definition})
readback = client._lro_json("POST", f"/workspaces/{TARGET_WORKSPACE_ID}/items/{monitoring_id}/getDefinition")["definition"]
actual = next(p for p in readback["parts"] if p["path"] == "MonitoringProperties.json")
assert json.loads(base64.b64decode(actual["payload"])) == properties
monitoring_database_name = TARGET_WORKSPACE_NAME + "_" + monitoring_name + "_Database"
deadline = time.monotonic() + 300
while True:
    databases = [i for i in client.list_items() if i["type"] == "KQLDatabase" and i["displayName"] == monitoring_database_name]
    if len(databases) == 1:
        monitoring_database_id = databases[0]["id"]
        break
    if len(databases) > 1 or time.monotonic() >= deadline:
        raise RuntimeError("Managed monitoring database not uniquely provisioned")
    time.sleep(10)
monitoring_database = client._json("GET", f"/workspaces/{TARGET_WORKSPACE_ID}/kqlDatabases/{monitoring_database_id}")
print("Managed monitoring configured:", monitoring_id)
print("Managed monitoring database:", monitoring_database_id)
print("Query endpoint:", monitoring_database["properties"]["queryServiceUri"])
print("Definition provisioning is not diagnostic collection acceptance. Complete and verify Workspace settings > Monitoring in the browser.")


## 3. Publish typed deployment variables

All references resolve to the current manual workspace.


In [ ]:
deployment = json.loads(json.dumps(inputs["deploymentTemplates"][DEPLOYMENT_STAGE]))
deployment.update(workspace_id=TARGET_WORKSPACE_ID, fabric_environment_id=TARGET_ENVIRONMENT_ID)
deployment["lakehouse"].update(lakehouse_id=TARGET_LAKEHOUSE_ID, files_root=DEPLOYMENT_FILES_ROOT)
sjd_names = {role:SJD_NAME_PREFIX + role for role in deployment["roles"]}
sjd_ids = resolve_sjd_ids(client, sjd_names)
for role,item_id in sjd_ids.items():
    deployment["roles"][role].update(display_name=sjd_names[role], item_id=item_id)
deployment_state_path = root_uri + "/deployment/deployment.json"
if notebookutils.fs.exists(deployment_state_path):
    previous_deployment = json.loads(notebookutils.fs.head(deployment_state_path, 1024 * 1024))
    for bundle in ("analytics_reporting", "operational_reporting"):
        deployment[bundle] = previous_deployment[bundle]
notebookutils.fs.mkdirs(root_uri + "/deployment")
save_deployment_state(notebookutils.fs, deployment_state_path, deployment)
print("Discovered deployment IDs retained:", deployment_state_path)
variable_tokens = {"DeploymentStage":DEPLOYMENT_STAGE,"WorkspaceId":TARGET_WORKSPACE_ID,"LakehouseId":TARGET_LAKEHOUSE_ID,"EnvironmentId":TARGET_ENVIRONMENT_ID}
variable_tokens.update({"SparkJob" + role.title() + "Id":item_id for role,item_id in sjd_ids.items()})
variable_text = inputs["variableTemplate"]
for name,value in variable_tokens.items():
    variable_text = variable_text.replace("{{ ." + name + " }}", value)
assert "{{" not in variable_text
variable_values = json.loads(variable_text)
variable_definition = {"format":"Default","parts":[{"path":name,"payloadType":"InlineBase64","payload":base64.b64encode(json.dumps(value,sort_keys=True).encode()).decode()} for name,value in (("settings.json",inputs["variableSettings"]),("variables.json",variable_values))]}
variable_library_id = ensure_item(client,"pc_dev_variables","VariableLibrary",variable_definition)
print("Variable library configured:", variable_library_id)
print("Typed deployment references:", [(v["name"],v["type"]) for v in variable_values["variables"]])


## 4. Publish the exact repository reporting definitions

Verify both semantic models and reports, then refresh both models and open both reports in the browser. Empty data is not a visual error, but it does not prove end-to-end processing.


In [ ]:
reporting_ids = {}
lakehouse_readback = client._json("GET",f"/workspaces/{TARGET_WORKSPACE_ID}/lakehouses/{TARGET_LAKEHOUSE_ID}")
base_tokens = {"TargetWorkspaceId":TARGET_WORKSPACE_ID,"TargetLakehouseId":TARGET_LAKEHOUSE_ID,"TargetLakehouseName":lakehouse_readback["displayName"]}
for bundle,label in (("analytics","analytics"),("operational","operations")):
    binding = deployment[bundle + "_reporting"]
    model_name = binding["semantic_model"]["target_name"]
    model_definition = render_definition(inputs["definitions"]["semantic-models/" + bundle],base_tokens)
    model_id = ensure_item(client,model_name,"SemanticModel",model_definition,approved_existing_id=binding["semantic_model"].get("target_item_id"))
    print("Semantic model definition verified:", model_name, model_id, flush=True)
    report_name = binding["report"]["target_name"]
    report_tokens = {"TargetSemanticModelId":model_id,"TargetSemanticModelName":model_name}
    report_definition = render_definition(inputs["definitions"]["reports/" + bundle],report_tokens,workspace_name=TARGET_WORKSPACE_NAME)
    report_id = ensure_item(client,report_name,"Report",report_definition,approved_existing_id=binding["report"].get("target_item_id"))
    binding["semantic_model"]["target_item_id"] = model_id
    binding["report"]["target_item_id"] = report_id
    reporting_ids[bundle] = {"semanticModelId":model_id,"reportId":report_id}
    print("Report definition verified:", report_name, report_id, flush=True)
save_deployment_state(notebookutils.fs, deployment_state_path, deployment)
print("Reporting bindings:", json.dumps(reporting_ids,sort_keys=True))


## 5. Publish RTI definitions

The Eventhouse and its automatically created database can share a name; item type is part of identity. The Queryset and dashboard use the managed monitoring database, not the separate operations database. Set APPROVED_QUERYSET_ID only to adopt an explicitly verified existing item.


In [ ]:
APPROVED_QUERYSET_ID = ""  # Set only after verifying an existing Queryset in this workspace.
operations_eventhouse_id = ensure_item(client,"pc_operations_eventhouse","Eventhouse")
print("Operations Eventhouse:",operations_eventhouse_id,flush=True)
operations_database_id = ensure_item(client,"pc_operations","KQLDatabase",creation_payload={"databaseType":"ReadWrite","parentEventhouseItemId":operations_eventhouse_id})
operations_database = client._json("GET",f"/workspaces/{TARGET_WORKSPACE_ID}/kqlDatabases/{operations_database_id}")
assert operations_database["properties"]["parentEventhouseItemId"] == operations_eventhouse_id
print("Operations database:",operations_database_id,flush=True)
rti_tokens = {"TargetWorkspaceId":TARGET_WORKSPACE_ID,"TargetKqlDatabaseId":monitoring_database_id,"TargetKqlQueryServiceUri":monitoring_database["properties"]["queryServiceUri"]}
queryset_definition = render_definition(inputs["definitions"]["rti/queryset"],rti_tokens)
queryset_id = ensure_item(client,"pc_dev_operations_queries","KQLQueryset",queryset_definition,approved_existing_id=APPROVED_QUERYSET_ID or None)
print("Operations Queryset definition verified:",queryset_id,flush=True)
dashboard_definition = render_definition(inputs["definitions"]["rti/dashboard"],rti_tokens)
dashboard_id = ensure_item(client,"pc_operations_dashboard","KQLDashboard",dashboard_definition)
print("Operations dashboard definition verified:",dashboard_id,flush=True)
print("RTI uses managed monitoring database:",monitoring_database_id)


## 6. Publish both operations rules, stopped with portal draft actions

Set APPROVED_ACTIVATOR_ID only after verifying the existing stopped manual Activator. Fabric requires a structurally valid action step, so the existing portal-authored draft action is retained, not the repository's old recipients. This is not recipient approval. Both rules stay stopped and the KQL source stays paused. No test action is sent. Review recipients, message fields, and channels before any separate handover approval.


In [ ]:
APPROVED_ACTIVATOR_ID = ""
if not APPROVED_ACTIVATOR_ID:
    raise ValueError("Create and verify the stopped portal Activator from section 13 first")
existing_definition = client._lro_json(
    "POST", f"/workspaces/{TARGET_WORKSPACE_ID}/items/{APPROVED_ACTIVATOR_ID}/getDefinition"
)["definition"]
existing_entities = json.loads(base64.b64decode(next(
    p for p in existing_definition["parts"] if p["path"] == "ReflexEntities.json"
)["payload"]))
draft_actions = []
for entity in existing_entities:
    rule = entity["payload"].get("definition", {})
    if rule.get("type") == "Rule":
        if rule.get("settings", {}).get("shouldRun") is not False:
            raise ValueError("Stop the existing rule before updating its definition")
        instance = json.loads(rule["instance"])
        draft_actions.extend(step["rows"] for step in instance["steps"] if step["name"] == "ActStep" and step["rows"])
if not draft_actions or len({json.dumps(rows, sort_keys=True) for rows in draft_actions}) != 1:
    raise ValueError("A unique portal-authored draft action requires operator review")
encoded_claims = client.token_provider.get_token().split(".")[1]
tenant_id = json.loads(base64.urlsafe_b64decode(encoded_claims + "=" * (-len(encoded_claims) % 4)))["tid"]
del encoded_claims
activator_tokens = {
    **rti_tokens,
    "TargetSparkJobDispatcherId": sjd_ids["dispatcher"],
    "TargetKqlDashboardId": dashboard_id,
}
activator_definition = stopped_activator_definition(
    inputs["definitions"]["activators/operations"],
    activator_tokens,
    tenant_id=tenant_id,
    dispatcher_name="pc_sjd_dev_dispatcher",
    action_rows=draft_actions[0],
)
operations_activator_id = ensure_item(
    client, "pc_dev_operations_activator", "Reflex", activator_definition,
    approved_existing_id=APPROVED_ACTIVATOR_ID or None,
)
print("Operations Activator definition verified:", operations_activator_id)
print("Both rules stopped; KQL source paused; portal draft actions retained without delivery approval.")
